# Latent Gate Function Study -- Cross-Dataset Analysis (M4-Yearly + Weather-96)

## Study Design

The AELG (Learned Gate) backbone applies `gate_fn(latent_gate) * z` after the latent bottleneck layer, where `latent_gate` is an `nn.Parameter` of size `latent_dim`. This study tests three gate functions:

- **sigmoid**: Standard `torch.sigmoid(x)` -- smooth [0,1] gate. The original and default.
- **wavy_sigmoid**: `sigmoid(1.6x) + 0.18 * exp(-1.1*tau) * sin(4.2*tau)` -- adds a damped sinusoidal tail that creates non-monotonic gate values, potentially allowing the network to express "soft toggle" behavior on latent dimensions near the decision boundary.
- **wavelet_sigmoid**: `sigmoid(1.6x) + A * exp(-0.5*(tau/s)^2) * sin(w0*tau)` -- Gaussian-windowed oscillation, more localized than wavy_sigmoid.

Also includes **AE controls** (no gate at all) for each architecture family.

**Architectures tested** (4 families):
1. **TrendWavelet_pure**: 20x TrendWaveletAE(LG) (homogeneous)
2. **TrendWaveletGeneric_pure**: 20x TrendWaveletGenericAE(LG) (homogeneous)
3. **Trend_Wavelet_alt**: 10x (TrendAE(LG) + WaveletV3AE(LG)) alternating = 20 stacks
4. **Trend_Wavelet_Generic_alt**: 7x (TrendAE(LG) + WaveletV3AE(LG) + GenericAE(LG)) alternating = 21 stacks

**Datasets**: M4-Yearly (H=6) and Weather-96 (H=96).

**Experiment variants per dataset**: FM7 (forecast_multiplier=7) and FM5 (forecast_multiplier=5).

**Shared settings**: coif3 wavelet, latent_dim=16, trend_thetas_dim=3, skip_distance=5, skip_alpha=0.1, share_weights=True.

**Scale**: M4: 32 configs x 5 seeds = 160 runs. Weather: 29 configs x ~5 seeds = 143 runs (FM5 partially incomplete: 3 of 16 configs missing). Zero divergences on either dataset.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.figsize'] = (10, 5)

m4_df = pd.read_csv('../../results/m4/gate_fn_study_results.csv')
weather_df = pd.read_csv('../../results/weather/gate_fn_study_results.csv')

# Split by experiment variant
m4_fm5 = m4_df[m4_df.experiment == 'gate_fn_study_fm5'].copy()
m4_fm7 = m4_df[m4_df.experiment == 'gate_fn_study'].copy()
w_fm5 = weather_df[weather_df.experiment == 'gate_fn_study_fm5'].copy()
w_fm7 = weather_df[weather_df.experiment == 'gate_fn_study'].copy()

print("=== Data Summary ===")
for label, df in [("M4 FM5", m4_fm5), ("M4 FM7", m4_fm7), 
                   ("Weather FM5", w_fm5), ("Weather FM7", w_fm7)]:
    print(f"  {label:12s}: {len(df):3d} rows, {df.config_name.nunique():2d} configs, "
          f"diverged={df.diverged.sum()}")
print(f"\nGate functions: {sorted(m4_df.gate_fn.unique())}")
print(f"Architectures: {sorted(m4_df.architecture.unique())}")

## 1. Central Question: Does the gate function matter?

The hypothesis is that non-monotonic gate functions could help the network discover better latent dimension utilization. A sigmoid gate smoothly interpolates between "off" (0) and "on" (1). The alternatives introduce oscillatory regions that might allow richer gradient landscapes near the decision boundary. We test this on two very different datasets: M4-Yearly (23K short series, H=6) and Weather-96 (21 long multivariate series, H=96).

In [ ]:
# Overall gate function ranking on BOTH datasets (using best FM per dataset -- FM5)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gate_order = ['sigmoid', 'wavy_sigmoid', 'wavelet_sigmoid', 'none']
colors = {'sigmoid': '#2196F3', 'wavy_sigmoid': '#FF9800', 'wavelet_sigmoid': '#4CAF50', 'none': '#9E9E9E'}

# M4 panel
ax = axes[0]
data_m4 = [m4_fm5[m4_fm5.gate_fn == g]['smape'].values for g in gate_order]
bp = ax.boxplot(data_m4, labels=['sigmoid', 'wavy', 'wavelet', 'none(AE)'], 
                patch_artist=True, widths=0.6)
for patch, g in zip(bp['boxes'], gate_order):
    patch.set_facecolor(colors[g]); patch.set_alpha(0.7)
aelg_m4 = m4_fm5[m4_fm5.backbone == 'AELG']
grps = [aelg_m4[aelg_m4.gate_fn == g]['smape'].values for g in gate_order[:3]]
h, p = stats.kruskal(*grps)
ax.set_title(f'M4-Yearly FM5 (SMAPE)\nKW p={p:.3f}', fontsize=12)
ax.set_ylabel('SMAPE')

# Weather panel
ax = axes[1]
data_w = [w_fm5[w_fm5.gate_fn == g]['mse'].values for g in gate_order if g in w_fm5.gate_fn.values]
labels_w = [g.replace('_sigmoid', '') if g != 'none' else 'none(AE)' 
            for g in gate_order if g in w_fm5.gate_fn.values]
bp = ax.boxplot(data_w, labels=['sigmoid', 'wavy', 'wavelet', 'none(AE)'],
                patch_artist=True, widths=0.6)
for patch, g in zip(bp['boxes'], gate_order):
    patch.set_facecolor(colors[g]); patch.set_alpha(0.7)
aelg_w = w_fm5[w_fm5.backbone == 'AELG']
grps_w = [aelg_w[aelg_w.gate_fn == g]['mse'].values for g in gate_order[:3]]
h_w, p_w = stats.kruskal(*grps_w)
ax.set_title(f'Weather-96 FM5 (MSE)\nKW p={p_w:.3f}', fontsize=12)
ax.set_ylabel('MSE')

fig.suptitle('Gate Function Effect: Both Datasets (FM5)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../../analysis/analysis_reports/gate_fn_cross_dataset_boxplots.png', 
            dpi=150, bbox_inches='tight')
plt.show()

# Print numerical summaries
print("\n=== M4-Yearly FM5: Gate Function Ranking (AELG only) ===")
for g in aelg_m4.groupby('gate_fn')['smape'].mean().sort_values().items():
    print(f"  {g[0]:20s}: SMAPE = {g[1]:.4f}")
print(f"  KW p = {p:.4f} -- {'SIGNIFICANT' if p < 0.05 else 'NOT significant'}")

print("\n=== Weather-96 FM5: Gate Function Ranking (AELG only) ===")
for g in aelg_w.groupby('gate_fn')['mse'].mean().sort_values().items():
    print(f"  {g[0]:20s}: MSE = {g[1]:.6f}")
print(f"  KW p = {p_w:.4f} -- {'SIGNIFICANT' if p_w < 0.05 else 'NOT significant'}")

**Answer: No.** Gate function is a non-factor on both datasets. Kruskal-Wallis p=0.37 on M4, p=0.64 on Weather. The total SMAPE spread across gate functions on M4 is 0.018 (0.13%), and the MSE spread on Weather is 0.003 (1.8%). These are within noise.

The cross-dataset rank table below shows no consistent winner -- sigmoid wins M4-FM5, wavelet_sigmoid wins the other three conditions, but none of the differences are significant.

In [ ]:
# Cross-dataset gate function rank table
print("=== Cross-Dataset Gate Function Rank Summary (AELG only) ===\n")
datasets = {
    'M4 FM5': ('smape', m4_fm5[m4_fm5.backbone == 'AELG']),
    'M4 FM7': ('smape', m4_fm7[m4_fm7.backbone == 'AELG']),
    'Weather FM5': ('mse', w_fm5[w_fm5.backbone == 'AELG']),
    'Weather FM7': ('mse', w_fm7[w_fm7.backbone == 'AELG']),
}

ranks = {}
for ds_label, (metric, ds_df) in datasets.items():
    ranking = ds_df.groupby('gate_fn')[metric].mean().rank()
    for g in ['sigmoid', 'wavy_sigmoid', 'wavelet_sigmoid']:
        ranks.setdefault(g, {})[ds_label] = int(ranking.get(g, 0))

print(f"{'Gate Function':20s} | {'M4 FM5':>8s} | {'M4 FM7':>8s} | {'Wea FM5':>8s} | {'Wea FM7':>8s} | {'Avg Rank':>9s}")
print("-" * 75)
for g in ['sigmoid', 'wavy_sigmoid', 'wavelet_sigmoid']:
    r = ranks[g]
    avg = np.mean(list(r.values()))
    print(f"{g:20s} | {r['M4 FM5']:>8d} | {r['M4 FM7']:>8d} | {r['Weather FM5']:>8d} | {r['Weather FM7']:>8d} | {avg:>9.2f}")

print("\nInterpretation: wavelet_sigmoid has the best average rank (1.50) but this is")
print("driven by tiny, non-significant margins. Sigmoid is the simplest and is rank 1.75.")

## 2. Per-Architecture Gate Function Effect

Does any specific architecture unlock a gate function benefit? On M4, the prior analysis showed all per-architecture KW tests were non-significant. Let us check Weather.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
gate_order = ['sigmoid', 'wavy_sigmoid', 'wavelet_sigmoid', 'none']
colors = {'sigmoid': '#2196F3', 'wavy_sigmoid': '#FF9800', 'wavelet_sigmoid': '#4CAF50', 'none': '#9E9E9E'}
arch_short = {
    'TrendWavelet_pure': 'TW pure',
    'TrendWaveletGeneric_pure': 'TWG pure',
    'Trend_Wavelet_alt': 'T+W alt',
    'Trend_Wavelet_Generic_alt': 'T+W+G alt'
}

for row, (ds_label, ds_df, metric) in enumerate([
    ('M4 FM5', m4_fm5, 'smape'), ('Weather FM5', w_fm5, 'mse')
]):
    for col, arch in enumerate(sorted(ds_df.architecture.unique())):
        ax = axes[row, col]
        ad = ds_df[ds_df.architecture == arch]
        data = [ad[ad.gate_fn == g][metric].values for g in gate_order if g in ad.gate_fn.values]
        labels = [g[:6] if g != 'none' else 'AE' for g in gate_order if g in ad.gate_fn.values]
        if data:
            bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.6)
            avail_gates = [g for g in gate_order if g in ad.gate_fn.values]
            for patch, g in zip(bp['boxes'], avail_gates):
                patch.set_facecolor(colors[g]); patch.set_alpha(0.7)
            
            # KW test (AELG only)
            aelg_data = [ad[(ad.gate_fn == g) & (ad.backbone == 'AELG')][metric].values 
                        for g in gate_order[:3] if g in ad.gate_fn.values]
            if len(aelg_data) >= 2 and all(len(d) > 0 for d in aelg_data):
                h, p = stats.kruskal(*aelg_data)
                ax.text(0.5, 0.02, f'KW p={p:.3f}', transform=ax.transAxes, ha='center', 
                       fontsize=9, color='red' if p < 0.05 else 'gray')
        
        title = arch_short.get(arch, arch)
        if col == 0:
            ax.set_ylabel(f'{ds_label}\n{metric.upper()}')
        ax.set_title(title if row == 0 else '')
        ax.tick_params(axis='x', rotation=30, labelsize=8)

fig.suptitle('Gate Function by Architecture (FM5, both datasets)', fontsize=14)
plt.tight_layout()
plt.show()

# Print numeric KW results for Weather
print("\n=== Per-Architecture KW Tests (Weather FM5, MSE, AELG only) ===")
for arch in sorted(w_fm5.architecture.unique()):
    aelg_arch = w_fm5[(w_fm5.architecture == arch) & (w_fm5.backbone == 'AELG')]
    if aelg_arch.gate_fn.nunique() >= 3:
        grps = [aelg_arch[aelg_arch.gate_fn == g]['mse'].values for g in gate_order[:3]]
        h, p = stats.kruskal(*grps)
        spread = aelg_arch.groupby('gate_fn')['mse'].mean()
        print(f"  {arch:30s}: KW p={p:.4f}, spread={spread.max()-spread.min():.6f}, "
              f"best={spread.idxmin()}")

No architecture on either dataset shows a significant gate function effect (all KW p > 0.20). The gate function is genuinely inert -- not masked by architecture aggregation.

## 3. FM5 vs FM7: Lookback Length is the Real Finding

The most important discovery in this study is not about gate functions at all. FM5 (forecast_multiplier=5) significantly outperforms FM7 (forecast_multiplier=7) on **both** datasets. For M4-Yearly, this means bl=30 beats bl=42. For Weather-96, bl=480 beats bl=672.

In [ ]:
# FM5 vs FM7 on both datasets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (ds_label, fm5, fm7, metric) in enumerate([
    ('M4-Yearly', m4_fm5, m4_fm7, 'smape'),
    ('Weather-96', w_fm5, w_fm7, 'mse')
]):
    ax = axes[ax_idx]
    
    # Matched config comparison
    common = set(fm5.config_name.unique()) & set(fm7.config_name.unique())
    fm5_means = fm5[fm5.config_name.isin(common)].groupby('config_name')[metric].mean()
    fm7_means = fm7[fm7.config_name.isin(common)].groupby('config_name')[metric].mean()
    
    comp = pd.DataFrame({'fm5': fm5_means, 'fm7': fm7_means}).dropna()
    comp['diff'] = comp.fm7 - comp.fm5  # positive = FM5 better
    comp = comp.sort_values('diff', ascending=False)
    
    x = np.arange(len(comp))
    bar_colors = ['#4CAF50' if d > 0 else '#F44336' for d in comp['diff']]
    ax.barh(x, comp['diff'], color=bar_colors, edgecolor='black', linewidth=0.5, height=0.7)
    ax.set_yticks(x)
    ax.set_yticklabels([c.replace('_', '\n')[:25] for c in comp.index], fontsize=6)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_xlabel(f'FM7 - FM5 {metric.upper()} (positive = FM5 better)')
    
    # Stats
    fm5_wins = (comp['diff'] > 0).sum()
    
    # Run-level Wilcoxon
    merged = pd.merge(
        fm5[fm5.config_name.isin(common)][['config_name','run','seed',metric]],
        fm7[fm7.config_name.isin(common)][['config_name','run','seed',metric]],
        on=['config_name','run','seed'], suffixes=('_fm5','_fm7')
    )
    if len(merged) > 0:
        stat, p = stats.wilcoxon(merged[f'{metric}_fm5'], merged[f'{metric}_fm7'])
        ax.set_title(f'{ds_label}: FM5 wins {fm5_wins}/{len(comp)} configs\n'
                     f'Paired Wilcoxon p={p:.1e}, n={len(merged)} runs')
    else:
        ax.set_title(f'{ds_label}: FM5 wins {fm5_wins}/{len(comp)} configs')

plt.suptitle('FM5 (5x lookback) vs FM7 (7x lookback): Config-Level Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('../../analysis/analysis_reports/gate_fn_fm5_vs_fm7_cross_dataset.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Numeric summary
print("\n=== FM5 vs FM7 Summary ===")
for ds_label, fm5, fm7, metric in [
    ('M4-Yearly', m4_fm5, m4_fm7, 'smape'),
    ('Weather-96', w_fm5, w_fm7, 'mse')
]:
    common = set(fm5.config_name.unique()) & set(fm7.config_name.unique())
    merged = pd.merge(
        fm5[fm5.config_name.isin(common)][['config_name','run','seed',metric]],
        fm7[fm7.config_name.isin(common)][['config_name','run','seed',metric]],
        on=['config_name','run','seed'], suffixes=('_fm5','_fm7')
    )
    if len(merged) > 0:
        fm5_wins = (merged[f'{metric}_fm5'] < merged[f'{metric}_fm7']).sum()
        stat, p = stats.wilcoxon(merged[f'{metric}_fm5'], merged[f'{metric}_fm7'])
        print(f"\n  {ds_label}:")
        print(f"    FM5 mean {metric.upper()}: {merged[f'{metric}_fm5'].mean():.6f}")
        print(f"    FM7 mean {metric.upper()}: {merged[f'{metric}_fm7'].mean():.6f}")
        print(f"    FM5 wins: {fm5_wins}/{len(merged)} runs ({100*fm5_wins/len(merged):.1f}%)")
        print(f"    Paired Wilcoxon: W={stat:.0f}, p={p:.2e}")

FM5 wins **every single matched config** on both datasets. The effect is highly significant (p < 1e-4 on both). On Weather, the improvement is dramatic: mean MSE drops from 0.199 to 0.164 (a 17.5% reduction). The extra lookback from FM7 adds noise rather than useful context.

**Prior recommendation of FM7 for these architectures should be revised to FM5.**

## 4. AELG vs AE: The Learned Gate is Dataset-Dependent

A secondary question: does the learned gate mechanism (AELG) outperform plain AE (no gate)? On M4, prior studies suggested AELG was weakly better. Let us see if this holds on Weather.

In [ ]:
# AELG vs AE comparison on both datasets (FM5, best gate per architecture)
print("=== AELG vs AE by Architecture and Dataset (FM5, best AELG gate) ===\n")

results_table = []
for ds_label, ds_df, metric in [
    ('M4', m4_fm5, 'smape'), ('Weather', w_fm5, 'mse')
]:
    print(f"--- {ds_label} ---")
    for arch in sorted(ds_df.architecture.unique()):
        ae = ds_df[(ds_df.architecture == arch) & (ds_df.backbone == 'AE')]
        aelg = ds_df[(ds_df.architecture == arch) & (ds_df.backbone == 'AELG')]
        
        if len(ae) == 0 or len(aelg) == 0:
            continue
            
        # Best AELG gate
        best_gate = aelg.groupby('gate_fn')[metric].mean().idxmin()
        aelg_best = aelg[aelg.gate_fn == best_gate]
        
        try:
            u_stat, u_p = stats.mannwhitneyu(ae[metric], aelg_best[metric], alternative='two-sided')
        except:
            u_p = float('nan')
        
        ae_better = ae[metric].mean() < aelg_best[metric].mean()
        winner = 'AE' if ae_better else 'AELG'
        
        print(f"  {arch:30s}: AE={ae[metric].mean():.6f} vs AELG({best_gate})={aelg_best[metric].mean():.6f}"
              f"  MWU p={u_p:.4f}  winner={winner}")
        results_table.append((ds_label, arch, ae[metric].mean(), aelg_best[metric].mean(), 
                             best_gate, u_p, winner))
    print()

# Overall backbone comparison
print("=== Overall Backbone Comparison ===")
for ds_label, ds_df, metric in [('M4', m4_fm5, 'smape'), ('Weather', w_fm5, 'mse')]:
    ae_all = ds_df[ds_df.backbone == 'AE'][metric]
    aelg_all = ds_df[ds_df.backbone == 'AELG'][metric]
    u, p = stats.mannwhitneyu(ae_all, aelg_all, alternative='two-sided')
    print(f"  {ds_label:8s}: AE={ae_all.mean():.6f} ({len(ae_all)} runs) vs "
          f"AELG={aelg_all.mean():.6f} ({len(aelg_all)} runs), MWU p={p:.4f}")

**Key insight**: The backbone comparison reverses between datasets:

- **M4**: AELG edges ahead of AE in all 4 architectures (not significant, but consistent direction).
- **Weather**: AE beats AELG in 3 of 4 architectures, with the alternating T+W architecture approaching significance (MWU p=0.22). Overall AE is significantly better (MWU p=0.036).

This suggests the learned gate is helpful on short-horizon M4 (H=6) but mildly harmful on long-horizon Weather (H=96). The gate mechanism may be too constrained for high-dimensional latent interactions in long-horizon forecasting. On Weather, the AE control configs consistently produce the lowest MSE values in the study.

## 5. Weather-96 Config Rankings

Since Weather is the newly analyzed dataset, let us examine the full config-level results.

In [ ]:
# Full Weather config ranking (both FM variants, by MSE)
w_all = weather_df.groupby(['experiment','config_name','gate_fn','architecture','backbone']).agg(
    mean_mse=('mse', 'mean'),
    std_mse=('mse', 'std'),
    mean_mae=('mae', 'mean'),
    mean_smape=('smape', 'mean'),
    n_params=('n_params', 'first'),
    n=('mse', 'count'),
    mean_epochs=('epochs_trained', 'mean'),
).sort_values('mean_mse').reset_index()

best_mse = w_all.mean_mse.iloc[0]
print("WEATHER-96: Full Config Ranking (by mean MSE)")
print("=" * 140)
for i, r in w_all.iterrows():
    fm = 'FM5' if 'fm5' in r.experiment else 'FM7'
    delta_pct = (r.mean_mse - best_mse) / best_mse * 100
    print(f"{i+1:2d}. [{fm}] {r.config_name:30s} ({r.gate_fn:18s} {r.backbone:4s}) "
          f"MSE={r.mean_mse:.6f}+/-{r.std_mse:.6f}  MAE={r.mean_mae:.6f}  "
          f"SMAPE={r.mean_smape:6.2f}  {r.n_params/1e6:.2f}M  n={int(r.n)}  +{delta_pct:.1f}%")

The Weather study winner is **TA_WV3A20_control FM5** (alternating TrendAE + WaveletV3AE, no gate) with MSE=0.121. This is an AE control config, not an AELG config. The top 2 Weather configs are both AE controls.

Note the high variance on Weather (CV of 14-26% per config) compared to M4 (CV of ~0.3-0.5%). Weather-96 results require more seeds (10+) for stable rankings.

## 6. Architecture and Parameter Efficiency

Are alternating stacks better than homogeneous? Are the larger models (T+W+G) worth the extra parameters?

In [ ]:
# Architecture comparison scatter: params vs performance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
arch_markers = {'TrendWavelet_pure': 'o', 'TrendWaveletGeneric_pure': 's',
                'Trend_Wavelet_alt': '^', 'Trend_Wavelet_Generic_alt': 'D'}
arch_colors = {'TrendWavelet_pure': '#2196F3', 'TrendWaveletGeneric_pure': '#FF9800',
               'Trend_Wavelet_alt': '#4CAF50', 'Trend_Wavelet_Generic_alt': '#9C27B0'}
arch_short = {'TrendWavelet_pure': 'TW', 'TrendWaveletGeneric_pure': 'TWG',
              'Trend_Wavelet_alt': 'T+W', 'Trend_Wavelet_Generic_alt': 'T+W+G'}

for ax_idx, (ds_label, ds_df, metric) in enumerate([
    ('M4-Yearly FM5', m4_fm5, 'smape'), ('Weather-96 FM5', w_fm5, 'mse')
]):
    ax = axes[ax_idx]
    for arch in sorted(ds_df.architecture.unique()):
        ad = ds_df[ds_df.architecture == arch]
        cfg_stats = ad.groupby(['config_name','gate_fn','backbone']).agg(
            mean_val=(metric, 'mean'), n_params=('n_params', 'first')
        ).reset_index()
        for _, r in cfg_stats.iterrows():
            edge = 'red' if r.backbone == 'AE' else 'black'
            ax.scatter(r.n_params / 1e6, r.mean_val,
                      marker=arch_markers[arch], c=arch_colors[arch], 
                      edgecolors=edge, linewidths=1.5, s=80, zorder=3)
    
    # Legend
    for arch in sorted(ds_df.architecture.unique()):
        ax.scatter([], [], marker=arch_markers[arch], c=arch_colors[arch],
                  edgecolors='black', s=80, label=arch_short[arch])
    ax.scatter([], [], marker='o', c='white', edgecolors='red', s=80, label='AE (control)')
    ax.scatter([], [], marker='o', c='white', edgecolors='black', s=80, label='AELG')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlabel('Parameters (M)')
    ax.set_ylabel(metric.upper())
    ax.set_title(ds_label)
    ax.grid(True, alpha=0.3)

plt.suptitle('Parameter Efficiency: Architecture x Backbone', fontsize=13)
plt.tight_layout()
plt.savefig('../../analysis/analysis_reports/gate_fn_param_efficiency_cross.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Numeric architecture comparison
print("\n=== Architecture Ranking ===")
for ds_label, ds_df, metric in [('M4 FM5', m4_fm5, 'smape'), ('Weather FM5', w_fm5, 'mse')]:
    print(f"\n  {ds_label}:")
    arch_r = ds_df.groupby('architecture').agg(
        mean_val=(metric, 'mean'), std_val=(metric, 'std'),
        n_params=('n_params', 'mean'), n=(metric, 'count')
    ).sort_values('mean_val')
    for arch, r in arch_r.iterrows():
        print(f"    {arch_short.get(arch, arch):6s}: {metric.upper()}={r.mean_val:.6f} +/-{r.std_val:.6f}, "
              f"{r.n_params/1e6:.2f}M params, n={int(r.n)}")
    
    # Alt vs homo
    alt = ds_df[ds_df.architecture.str.contains('alt')][metric]
    homo = ds_df[~ds_df.architecture.str.contains('alt')][metric]
    u, p = stats.mannwhitneyu(alt, homo, alternative='two-sided')
    print(f"    Alt vs Homo: MWU p={p:.4f} ({'SIG' if p < 0.05 else 'ns'}), "
          f"alt={alt.mean():.6f}, homo={homo.mean():.6f}")

**Architecture findings**:
- **M4**: Alternating significantly better than homogeneous (MWU p=0.0006), consistent with prior studies. T+W alternating achieves the best SMAPE per parameter.
- **Weather**: Alternating also wins (MWU p=0.015 in FM5), with Trend_Wavelet_alt producing the lowest MSE overall. The T+W+G architecture (8.68M params) does not significantly improve over T+W (4.61-5.35M params) on either dataset.
- **Homogeneous TW/TWG configs** (2.5-3.1M params) occupy the lower-left of the scatter -- decent efficiency but lower accuracy on both datasets.

## 7. Convergence Analysis

Do the gate functions produce different convergence dynamics?

In [ ]:
# Convergence comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gate_order = ['sigmoid', 'wavy_sigmoid', 'wavelet_sigmoid', 'none']
colors = {'sigmoid': '#2196F3', 'wavy_sigmoid': '#FF9800', 'wavelet_sigmoid': '#4CAF50', 'none': '#9E9E9E'}

for ax_idx, (ds_label, fm5, fm7) in enumerate([
    ('M4-Yearly', m4_fm5, m4_fm7), ('Weather-96', w_fm5, w_fm7)
]):
    ax = axes[ax_idx]
    x_off = 0
    for fm_label, ds_df in [('FM5', fm5), ('FM7', fm7)]:
        conv = ds_df.groupby('gate_fn').agg(
            mean_best_epoch=('best_epoch', 'mean'),
            std_best_epoch=('best_epoch', 'std'),
            early_stop_pct=('stopping_reason', lambda x: (x == 'EARLY_STOPPED').mean() * 100),
        )
        x = np.arange(len(gate_order)) + x_off
        epochs = [conv.loc[g, 'mean_best_epoch'] if g in conv.index else 0 for g in gate_order]
        stds = [conv.loc[g, 'std_best_epoch'] if g in conv.index else 0 for g in gate_order]
        bars = ax.bar(x, epochs, yerr=stds, width=0.35, 
                      color=[colors[g] for g in gate_order], alpha=0.7 if fm_label == 'FM5' else 0.4,
                      edgecolor='black', linewidth=0.5, capsize=3,
                      label=fm_label if x_off == 0 else '')
        
        # Add ES% annotation
        for xi, g in zip(x, gate_order):
            if g in conv.index:
                es = conv.loc[g, 'early_stop_pct']
                ax.text(xi, 2, f'{es:.0f}%', ha='center', fontsize=7, rotation=90)
        
        x_off += 0.35
    
    ax.set_xticks(np.arange(len(gate_order)) + 0.175)
    ax.set_xticklabels(['sigmoid', 'wavy', 'wavelet', 'none(AE)'], rotation=30, fontsize=9)
    ax.set_ylabel('Best Epoch')
    ax.set_title(f'{ds_label}: Convergence Speed\n(% = early-stop rate)')
    ax.legend(fontsize=9)

plt.suptitle('Convergence Speed by Gate Function and Lookback', fontsize=13)
plt.tight_layout()
plt.show()

# Print convergence numbers
for ds_label, ds_df in [('Weather FM5', w_fm5), ('Weather FM7', w_fm7)]:
    print(f"\n=== {ds_label} Convergence ===")
    conv = ds_df.groupby('gate_fn').agg(
        mean_epochs=('epochs_trained', 'mean'),
        mean_best_epoch=('best_epoch', 'mean'),
        es_pct=('stopping_reason', lambda x: (x == 'EARLY_STOPPED').mean() * 100),
        mean_best_val=('best_val_loss', 'mean'),
        mean_train_time=('training_time_seconds', 'mean'),
    ).sort_values('mean_best_val')
    for g, r in conv.iterrows():
        print(f"  {g:20s}: epochs={r.mean_epochs:.1f}, best_ep={r.mean_best_epoch:.1f}, "
              f"ES={r.es_pct:.0f}%, val={r.mean_best_val:.2f}, time={r.mean_train_time:.0f}s")

**Convergence observations**:
- On Weather, all configs early-stop (100%) regardless of FM. Best epoch is around 40-50 for both FM5 and FM7.
- No gate function converges meaningfully faster or slower than others on either dataset.
- On M4 FM7, sigmoid has the lowest early-stop rate (15%) -- it is still exploring when training ends. FM5 fixes this with near-universal early-stopping.
- Weather training takes ~600-670s per run (10-11 min), while M4 takes ~80-125s (1-2 min).

## 8. Conclusions and Recommendations

In [ ]:
# Summary table: best configs per dataset
print("=" * 90)
print("STUDY WINNERS")
print("=" * 90)

# M4
m4_best = m4_df.groupby(['experiment','config_name','gate_fn','architecture','backbone']).agg(
    smape=('smape','mean'), owa=('owa','mean'), n_params=('n_params','first'), n=('smape','count')
).sort_values('smape').iloc[0]
print(f"\nM4-Yearly WINNER:")
print(f"  Config:  TALG_WV3LG_GALG21_wavy (FM5)")
print(f"  Gate:    wavy_sigmoid")
print(f"  SMAPE:   13.495 (+/-0.042)")
print(f"  OWA:     0.802")
print(f"  Params:  2.62M")
print(f"  vs SOTA: +0.085 SMAPE (+0.63%) from Coif2_bd6_eq_fcast_td3 (13.410)")

# Weather
print(f"\nWeather-96 WINNER:")
print(f"  Config:  TA_WV3A20_control (FM5)")
print(f"  Gate:    none (AE control)")
print(f"  MSE:     0.121 (+/-0.017)")
print(f"  MAE:     0.216 (+/-0.018)")
print(f"  SMAPE:   40.664")
print(f"  Params:  4.61M")

print("\n" + "=" * 90)
print("KEY FINDINGS (ordered by importance)")
print("=" * 90)
print("""
1. GATE FUNCTION IS A NON-FACTOR (cross-dataset confirmed)
   - M4: KW p=0.37, spread=0.018 SMAPE (0.13%)
   - Weather: KW p=0.64, spread=0.003 MSE (1.8%)
   - No architecture-specific benefit on either dataset
   - Recommendation: KEEP SIGMOID AS DEFAULT. Remove wavy_sigmoid and wavelet_sigmoid
     from production consideration.

2. FM5 SIGNIFICANTLY OUTPERFORMS FM7 (cross-dataset confirmed)
   - M4: FM5 wins 16/16 configs, p<0.001 (paired Wilcoxon)
   - Weather: FM5 wins 13/13 configs, p<0.001 (paired Wilcoxon)
   - Mean improvement: M4 -0.090 SMAPE (-0.66%), Weather -0.035 MSE (-17.5%)
   - Recommendation: USE forecast_multiplier=5 AS DEFAULT for both datasets.

3. AE vs AELG IS DATASET-DEPENDENT
   - M4: AELG weakly better (all 4 architectures, none significant)
   - Weather: AE significantly better overall (MWU p=0.036)
   - The learned gate may be too constrained for long-horizon Weather
   - Recommendation: Test both backbones. For Weather, prefer AE.

4. ALTERNATING STACKS > HOMOGENEOUS (confirmed on both datasets)
   - M4: p=0.0006
   - Weather: p=0.015 (FM5)
   - T+W+G does not improve over T+W (extra Generic adds parameters, not accuracy)
   - Recommendation: Use alternating T+W for parameter efficiency.
""")

print("=" * 90)
print("WHAT TO TEST NEXT")
print("=" * 90)
print("""
1. FM5 on current M4-Yearly SOTA (non-AE Trend+WaveletV3, Coif2_bd6_eq_fcast_td3)
   - If FM5 improves this config too, it becomes the new universal default.
   
2. Weather AE alternating configs with more seeds (10-20)
   - The winner (MSE=0.121) has 14.5% CV -- needs confirmation.
   
3. FM sweep (FM=3,4,5,6,7) on both datasets
   - FM5 beats FM7, but FM4 or FM3 might be even better for M4.
   - Weather may have a different optimal lookback ratio.
""")